In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Sgoff\.cache\kagglehub\datasets\iarunava\cell-images-for-detecting-malaria\versions\1


In [2]:
import os
import cv2
import numpy as np

# your kagglehub path
base_path = r"C:\Users\Sgoff\.cache\kagglehub\datasets\iarunava\cell-images-for-detecting-malaria\versions\1"

# go one level deeper
path = os.path.join(base_path, "cell_images")

data = []
labels = []

for label in ["Parasitized", "Uninfected"]:
    folder = os.path.join(path, label)
    class_label = 1 if label == "Parasitized" else 0
    
    print("Reading:", folder)  # debug line
    
    for img in os.listdir(folder):
        img_path = os.path.join(folder, img)
        
        image = cv2.imread(img_path)
        if image is None:
            continue  # skip bad images
        
        image = cv2.resize(image, (64,64))
        
        data.append(image.flatten())
        labels.append(class_label)

X = np.array(data) / 255.0
y = np.array(labels)

print("Data loaded:", X.shape)

Reading: C:\Users\Sgoff\.cache\kagglehub\datasets\iarunava\cell-images-for-detecting-malaria\versions\1\cell_images\Parasitized
Reading: C:\Users\Sgoff\.cache\kagglehub\datasets\iarunava\cell-images-for-detecting-malaria\versions\1\cell_images\Uninfected
Data loaded: (27558, 12288)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [5]:
from sklearn.linear_model import LogisticRegression

models = {
    "Logistic": LogisticRegression(max_iter=3000, solver='saga'),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(),
    "RandomForest": RandomForestClassifier()
}

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

predictions = {}
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    
    pred = model.predict(X_test)
    
    predictions[name] = pred
    
    results[name] = {
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred)
    }

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

ann = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

ann.compile(optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy'])

ann.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

pred_ann = (ann.predict(X_test) > 0.5).astype(int)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results["ANN"] = {
    "accuracy": accuracy_score(y_test, pred_ann),
    "precision": precision_score(y_test, pred_ann),
    "recall": recall_score(y_test, pred_ann),
    "f1": f1_score(y_test, pred_ann)
}

predictions["ANN"] = pred_ann

In [ ]:
for model, metrics in results.items():
    print(f"\n{model}")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

In [ ]:
import matplotlib.pyplot as plt

names = list(results.keys())
acc = [results[m]["accuracy"] for m in names]

plt.bar(names, acc)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.xticks(rotation=45)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

for name, pred in predictions.items():
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm)
    
    disp.plot()
    plt.title(name)
    plt.show()